# single variable_data.csv 생성
'2023 노인실태조사_DATA','요양기관 동부읍면부_추가'를 바탕으로 전처리 진행 후, 생성한 총 19개의 독거노인 단일변수(0~1 표준화 진행) dataset

In [1]:
import pandas as pd
import numpy as np

# 데이터 로드
df = pd.read_csv('2023년 노인실태조사_DATA.csv', low_memory=False)

# 1인 가구(독거노인) 데이터 필터링
df_solo = df[df['HTYPE'] == 1].copy()

# 필터링 결과 확인 (EDA)
print(f"전체 응답자 수: {len(df):,}명")
print(f"독거노인(1인가구) 필터링 완료: {len(df_solo):,}명")

전체 응답자 수: 10,078명
독거노인(1인가구) 필터링 완료: 3,448명


## 1. 건강 관련 단일변수

In [2]:
# 우울증 (risk_depression)
dep_cols = [f'B8_{i}' for i in range(1, 16)]
df_solo[dep_cols] = df_solo[dep_cols].replace([9, 99], np.nan)
pos_items = ['B8_1', 'B8_5', 'B8_7', 'B8_11', 'B8_13']
neg_items = [col for col in dep_cols if col not in pos_items]
for col in pos_items: df_solo[col + '_score'] = df_solo[col] - 1
for col in neg_items: df_solo[col + '_score'] = 2 - df_solo[col]
score_cols = [col + '_score' for col in dep_cols]
df_solo['risk_depression'] = df_solo[score_cols].sum(axis=1) / 15.0

In [3]:
# 처방 약물 수 (risk_drug)
df_solo['B4'] = pd.to_numeric(df_solo['B4'], errors='coerce').fillna(0)
df_solo['B4_clipped'] = df_solo['B4'].clip(upper=10)
df_solo['risk_drug'] = df_solo['B4_clipped'] / 10.0

In [4]:
# IADL (risk_iadl)
iadl_cols = [f'C8_1_{i}' for i in range(1, 11)]
df_solo[iadl_cols] = df_solo[iadl_cols].replace([9, 99], np.nan)
for col in iadl_cols: df_solo[col + '_deficit'] = (df_solo[col] - 1) / 2.0
deficit_cols = [col + '_deficit' for col in iadl_cols]
df_solo['risk_iadl'] = df_solo[deficit_cols].sum(axis=1) / 10.0

# 1.0 초과값을 1.0으로 강제 조정 (Clipping)
df_solo['risk_iadl'] = df_solo['risk_iadl'].clip(upper=1.0)

In [5]:
# 주관적 건강 상태 (risk_health)
df_solo['B1'] = df_solo['B1'].replace([9, 99], np.nan)
df_solo['risk_health'] = (df_solo['B1'] - 1) / 4.0

In [6]:
# 연령 (risk_age)
df_solo['risk_age'] = (df_solo['RES_AGE'] - 65) / (100 - 65)
df_solo['risk_age'] = df_solo['risk_age'].clip(0, 1)

## 2. 경제 관련 단일변수

In [7]:
# 주택 점유 형태 (risk_housing_tenure)
df_solo['risk_housing_tenure'] = np.where(df_solo['H1'].isin([3, 4]), 1.0, 0.0)

In [8]:
# 근로 여부 (risk_unemployed)
df_solo['risk_unemployed'] = np.where(df_solo['E1'].isin([2, 3]), 1.0, 0.0)

In [9]:
# 경제 상태 만족도 (risk_subj_econ)
df_solo['H15_2'] = df_solo['H15_2'].replace([9, 99], np.nan)
df_solo['risk_subj_econ'] = (df_solo['H15_2'] - 1) / 4.0

## 3. 주거 관련 단일변수

In [10]:
# 주택 내 안전 설비 부재 (risk_housing_safety)
safety_risk_cols = []
for i in range(1, 9):
    c_in = f'H3_1_{i}'; c_nd = f'H3_2_{i}'; r_col = f'risk_safety_{i}'
    safety_risk_cols.append(r_col)
    df_solo[r_col] = np.nan
    df_solo.loc[df_solo[c_in] == 1, r_col] = 0.0
    df_solo.loc[(df_solo[c_in] == 2) & (df_solo[c_nd] == 2), r_col] = 0.5
    df_solo.loc[(df_solo[c_in] == 2) & (df_solo[c_nd] == 1), r_col] = 1.0
df_solo['risk_housing_safety'] = df_solo[safety_risk_cols].mean(axis=1)

In [11]:
# 주거환경 불편함 (risk_housing_env)
df_solo['H2'] = df_solo['H2'].replace([9, 99], np.nan)
df_solo['risk_housing_env'] = (df_solo['H2'] - 1) / 4.0

## 4. 사회적 활동 관련 단일변수

In [12]:
# 단체 활동 미참여 (risk_group)
risk_map = {0: 1.0, 6: 0.83, 5: 0.67, 4: 0.50, 3: 0.33, 2: 0.17, 1: 0.0, 9: np.nan, 99: np.nan}
for i in range(1, 4): df_solo[f'D5_{i}_risk'] = df_solo[f'D5_{i}'].map(risk_map)
df_solo['risk_group'] = df_solo[['D5_1_risk', 'D5_2_risk', 'D5_3_risk']].min(axis=1)

In [13]:
# 외출/대면 교류 빈도 (risk_meet)
df_solo['risk_meet'] = np.nan
cond_has_friend = (df_solo['F6_1'] == 1)
df_solo.loc[cond_has_friend, 'risk_meet'] = (df_solo.loc[cond_has_friend, 'F6_2'] - 1) / 6.0
df_solo.loc[df_solo['F6_1'] == 2, 'risk_meet'] = 1.0

In [14]:
# 디지털 정보화 단절 (risk_digital)
df_solo['risk_digital'] = np.nan
cond_smart = (df_solo['D11_1_1'] == 1) & (df_solo['D11_2_1'] == 1)
df_solo.loc[cond_smart & (df_solo['D12_1'] == 1) & (df_solo['D12_2'] == 1), 'risk_digital'] = 0.0
cond_one = cond_smart & (((df_solo['D12_1'] == 1) & (df_solo['D12_2'] == 2)) | ((df_solo['D12_1'] == 2) & (df_solo['D12_2'] == 1)))
df_solo.loc[cond_one, 'risk_digital'] = 0.35
df_solo.loc[cond_smart & (df_solo['D12_1'] == 2) & (df_solo['D12_2'] == 2), 'risk_digital'] = 0.75
df_solo.loc[(df_solo['D11_1_1'] == 2) | (df_solo['D11_2_1'] == 2), 'risk_digital'] = 1.0

## 5. 행태 및 심리 관련 단일변수

In [15]:
# 영양 방임 (risk_nutrition)
for col in ['B13_2', 'B13_6', 'B13_7', 'B13_10']: df_solo[col + '_risk'] = 2.0 - df_solo[col]
df_solo['risk_nutrition'] = df_solo[[c + '_risk' for c in ['B13_2', 'B13_6', 'B13_7', 'B13_10']]].mean(axis=1)

In [16]:
# 흡연 (risk_smoke)
df_solo['risk_smoke'] = np.where(df_solo['B10'] == 1, 1.0, 0.0)

In [17]:
# 음주 (risk_alcohol)
df_solo['risk_alcohol'] = 0.0
df_solo.loc[df_solo['B11'].isin([5, 6]), 'risk_alcohol'] = 0.5
df_solo.loc[(df_solo['B11'] == 7) | (df_solo['B11_1a'].fillna(0) >= 5), 'risk_alcohol'] = 1.0

In [18]:
# 자살 생각 (risk_suicide)
df_solo['risk_suicide'] = np.where(df_solo['B9'] == 1, 1.0, 0.0)

In [19]:
# 연락망 단절 (risk_network)
def calc_net(exist_col, freq_col):
    res = pd.Series(np.nan, index=df_solo.index)
    res.loc[df_solo[exist_col] == 1] = (df_solo.loc[df_solo[exist_col] == 1, freq_col] - 1) / 6.0
    res.loc[df_solo[exist_col] == 2] = 1.0
    return res
df_solo['risk_network'] = pd.DataFrame({
    'f2': calc_net('F2_1', 'F2_3'), 'f4': calc_net('F4_1', 'F4_3'),
    'f5': calc_net('F5_1', 'F5_3'), 'f6': calc_net('F6_1', 'F6_3')
}).min(axis=1)

## '요양기관 동부 읍면부_추가.csv' 활용 변수

In [20]:
import pandas as pd
import numpy as np

# [1단계] 번역 사전 적용 및 df_solo에 마스터 키(region_key) 생성
sido_translator = {
    11: '서울특별시', 21: '부산광역시', 22: '대구광역시', 23: '인천광역시',
    24: '광주광역시', 25: '대전광역시', 26: '울산광역시', 29: '세종특별자치시',
    31: '경기도', 32: '강원특별자치도', 33: '충청북도', 34: '충청남도', 
    35: '전북특별자치도', 36: '전라남도', 37: '경상북도', 38: '경상남도', 39: '제주특별자치도'
}

df_solo = df_solo.copy()

df_solo['TSIDO_num'] = pd.to_numeric(df_solo['TSIDO'], errors='coerce')
df_solo['시도명'] = df_solo['TSIDO_num'].map(sido_translator)

df_solo['TareaUMD_2_num'] = pd.to_numeric(df_solo['TareaUMD_2'], errors='coerce')
df_solo['동부읍면부'] = df_solo['TareaUMD_2_num'].map({1: '동부', 2: '읍면부'})

df_solo['region_key'] = df_solo['시도명'] + "_" + df_solo['동부읍면부']

In [21]:
# 외부 데이터(요양기관, 인구) 불러오기 및 집계

# 요양기관 데이터 처리
df_infra = pd.read_csv('요양기관_동부읍면부_추가.csv', encoding='utf-8-sig')
df_infra['region_key'] = df_infra['시도명'] + "_" + df_infra['동부읍면부']
infra_counts = df_infra.groupby('region_key').size().reset_index(name='infra_count')

# 인구수 데이터 처리
df_pop = pd.read_csv('행정안전부_지역별(행정동) 성별 연령별 주민등록 인구수_20260430.csv', encoding='cp949')
df_pop['동부읍면부'] = np.where(df_pop['읍면동명'].str.endswith(('읍', '면')), '읍면부', '동부')
df_pop['region_key'] = df_pop['시도명'] + "_" + df_pop['동부읍면부']

# 65세 이상 인구 지능형 추출
elderly_cols = [col for col in df_pop.columns if '세' in col and ('남자' in col or '여자' in col) and int(col.split('세')[0]) >= 65]
df_pop['elderly_pop'] = df_pop[elderly_cols].sum(axis=1)

# 총 인구수 추출
df_pop['total_pop'] = df_pop['계'].astype(str).str.replace(',', '').astype(float)
pop_stats = df_pop.groupby('region_key')[['total_pop', 'elderly_pop']].sum().reset_index()

In [22]:
# df_solo에 모두 합치고 복합변수(infra_gap, 고령화율) 계산

# 데이터 병합
df_solo = df_solo.merge(infra_counts, on='region_key', how='left')
df_solo = df_solo.merge(pop_stats, on='region_key', how='left')

# 결측치 방어 (요양기관 없는 곳 0처리, 분모 0 방지용 +1)
df_solo['infra_count'] = df_solo['infra_count'].fillna(0)

df_solo['risk_infra_gap'] = df_solo['infra_count'] / (df_solo['elderly_pop'] + 1)
df_solo['risk_aging_rate'] = df_solo['elderly_pop'] / (df_solo['total_pop'] + 1)

# Min-Max 스케일링 (0~1 정규화)
for col in ['risk_infra_gap', 'risk_aging_rate']:
    df_solo[col] = (df_solo[col] - df_solo[col].min()) / (df_solo[col].max() - df_solo[col].min())

## 최종 Dataframe 생성

In [23]:
# 최종 클러스터링에 사용할 변수만 추출하여 새로운 데이터프레임 생성
final_features = [
    'risk_depression', 'risk_drug', 'risk_iadl', 'risk_health', 'risk_age',
    'risk_housing_tenure', 'risk_unemployed', 'risk_subj_econ', 
    'risk_housing_safety', 'risk_housing_env', 'risk_group', 'risk_meet', 
    'risk_digital', 'risk_nutrition', 'risk_smoke', 'risk_alcohol', 
    'risk_suicide', 'risk_network',
    'risk_infra_gap', 'risk_aging_rate'  
]

df_clustering = df_solo[final_features].copy()

## 결측치 확인 및 표시

In [24]:
for col in df_clustering.columns:
    if df_clustering[col].isnull().sum() > 0:
        # 1. "원래 비어있었다"는 표시 (0: 원래 값 있음, 1: 원래 결측치)
        df_clustering[f'{col}_is_missing'] = df_clustering[col].isnull().astype(int)
        
        # 2. 클러스터링 알고리즘 에러 방지를 위해 결측치(NaN)를 중앙값으로 채우기
        df_clustering[col] = df_clustering[col].fillna(df_clustering[col].median())

## StadardScaler 적용

In [25]:
from sklearn.preprocessing import StandardScaler

# 1. 0~1로 정규화된 위험도 변수들(risk_...) 선택 (결측치 표시 컬럼 제외)
target_cols = [col for col in df_clustering.columns if col.startswith('risk_') and 'missing' not in col]

# 2. StandardScaler 적용하여 평균 0, 분산 1 데이터 생성
scaler = StandardScaler()
scaled_values = scaler.fit_transform(df_clustering[target_cols])

# 3. 덮어씌우지 말고, 데이터프레임에 '_std' 꼬리표를 단 컬럼으로 추가
std_cols = [col + '_std' for col in target_cols]
df_clustering[std_cols] = scaled_values

# 4. CSV 파일로 최종 추출 (0~1 해석용 변수 + _std 학습용 변수 모두 포함)
df_clustering.to_csv('single variable_data.csv', index=False, encoding='utf-8-sig')
print("✅ 전처리 완벽 종료: 해석용(0~1) 및 K-Means용(std) 변수 통합 저장 완료!")

✅ 전처리 완벽 종료: 해석용(0~1) 및 K-Means용(std) 변수 통합 저장 완료!


## CSV 파일 추출

In [202]:
df_clustering.to_csv('single variable_data.csv', index=False, encoding='utf-8-sig')

print("전처리 완료 및 CSV 파일 저장 완료")

전처리 완료 및 CSV 파일 저장 완료


## 결측치 처리 검토

In [176]:
import pandas as pd

df_final = pd.read_csv('single variable_data.csv')

# 결측치 현황 요약
missing_info = pd.DataFrame({
    '변수명 (Column)': df_final.columns,
    '결측치 개수 (Missing)': df_final.isnull().sum().values,
    '결측치 비율 (%)': (df_final.isnull().sum().values / len(df) * 100).round(2)
})

print("=== 각 변수별 결측치 현황 ===")
print(missing_info.to_string(index=False))

=== 각 변수별 결측치 현황 ===
               변수명 (Column)  결측치 개수 (Missing)  결측치 비율 (%)
            risk_depression                 0         0.0
                  risk_drug                 0         0.0
                  risk_iadl                 0         0.0
                risk_health                 0         0.0
                   risk_age                 0         0.0
        risk_housing_tenure                 0         0.0
            risk_unemployed                 0         0.0
             risk_subj_econ                 0         0.0
        risk_housing_safety                 0         0.0
           risk_housing_env                 0         0.0
                 risk_group                 0         0.0
                  risk_meet                 0         0.0
               risk_digital                 0         0.0
             risk_nutrition                 0         0.0
                 risk_smoke                 0         0.0
               risk_alcohol                 0      